In [1]:
"""
MSDS 422 Final Project - Milestone 2
Exploratory Data Analysis
Hospital Readmission Prediction for Diabetes Patients

Dataset: Diabetes 130-US Hospitals (UCI Machine Learning Repository)

Author: Feifan Liu, Jiaohao Li, Abdullah Abdul Sami
Date: March 2026
"""



'\nMSDS 422 Final Project - Milestone 2\nExploratory Data Analysis\nHospital Readmission Prediction for Diabetes Patients\n\nDataset: Diabetes 130-US Hospitals (UCI Machine Learning Repository)\n\nAuthor: Feifan Liu, Jiaohao Li, Abdullah Abdul Sami\nDate: March 2026\n'

In [2]:
"""
02_feature_engineering.py
=========================
Feature Engineering for Diabetes 130-US Hospitals Readmission Prediction

This script handles data preprocessing and feature engineering, including:
- Data cleaning and filtering
- Missing value handling
- Time-based feature creation (length of stay bins, prior visit counts)
- Categorical encoding
- Feature scaling
- Train/test split and export

Author: Jiahao Li
Team: Final Project - G4 (Abdullah, Jiahao, Feifan)
Course: MSDS 422 - Practical Machine Learning
Date: February 2026
"""

import os
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

warnings.filterwarnings("ignore")

# ============================================================
# Configuration
# ============================================================
RAW_DATA_PATH = os.path.join("data", "diabetic_data.csv")
OUTPUT_DIR = os.path.join("data", "processed")
RANDOM_STATE = 42
TEST_SIZE = 0.2


def load_raw_data(filepath):
    """Load the raw UCI diabetes dataset."""
    print(f"Loading raw data from {filepath}...")
    df = pd.read_csv(filepath, na_values="?")
    print(f"  Raw dataset shape: {df.shape}")
    print(f"  Encounters: {len(df):,}")
    print(f"  Features: {df.shape[1]}")
    return df


def clean_data(df):
    """
    Clean dataset by removing invalid records and duplicates.

    Steps:
    1. Remove duplicate patient encounters (keep first)
    2. Remove deceased/hospice discharges (not readmission candidates)
    3. Drop high-missing-value columns (weight, payer_code)
    """
    print("\n--- Data Cleaning ---")
    initial_count = len(df)

    # Keep only the first encounter per patient
    df = df.sort_values("encounter_id").drop_duplicates(
        subset=["patient_nbr"], keep="first"
    )
    print(f"  After dedup (first encounter per patient): {len(df):,} "
          f"(removed {initial_count - len(df):,})")

    # Remove deceased/hospice discharges (discharge_disposition_id in [11,13,14,19,20,21])
    invalid_discharges = [11, 13, 14, 19, 20, 21]
    before = len(df)
    df = df[~df["discharge_disposition_id"].isin(invalid_discharges)]
    print(f"  After removing invalid discharges: {len(df):,} "
          f"(removed {before - len(df):,})")

    # Drop columns with excessive missing values
    drop_cols = ["weight", "payer_code", "encounter_id", "patient_nbr"]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    print(f"  Dropped columns: {[c for c in drop_cols if c != 'encounter_id']}")

    print(f"  Final cleaned shape: {df.shape}")
    return df


def create_target_variable(df):
    """Create binary readmission target variable."""
    print("\n--- Target Variable ---")
    df["readmit_binary"] = (df["readmitted"] == "<30").astype(int)

    pos = df["readmit_binary"].sum()
    neg = len(df) - pos
    print(f"  Positive class (readmitted <30 days): {pos:,} ({pos/len(df)*100:.1f}%)")
    print(f"  Negative class: {neg:,} ({neg/len(df)*100:.1f}%)")
    print(f"  Imbalance ratio: {neg/pos:.1f}:1")

    df = df.drop(columns=["readmitted"])
    return df


def engineer_time_based_features(df):
    """
    Create time-based and utilization features.

    Features created:
    - los_bin: Length of stay bins (short/medium/long/extended)
    - total_prior_visits: Sum of outpatient + emergency + inpatient visits
    - prior_visit_intensity: Categorical intensity of prior visits
    - visit_ratio_emergency: Proportion of emergency visits among prior visits
    - visit_ratio_inpatient: Proportion of inpatient visits among prior visits
    - num_medications_bin: Medication count bins
    - high_utilizer: Flag for patients with 3+ prior visits
    - procedure_intensity: Ratio of procedures to time in hospital
    """
    print("\n--- Time-Based & Utilization Features ---")

    # Length of stay bins
    df["los_bin"] = pd.cut(
        df["time_in_hospital"],
        bins=[0, 2, 5, 9, 14],
        labels=["short", "medium", "long", "extended"],
        right=True
    )
    print(f"  los_bin distribution:\n{df['los_bin'].value_counts().to_string()}")

    # Total prior visits (outpatient + emergency + inpatient)
    df["total_prior_visits"] = (
        df["number_outpatient"]
        + df["number_emergency"]
        + df["number_inpatient"]
    )

    # Prior visit intensity categories
    df["prior_visit_intensity"] = pd.cut(
        df["total_prior_visits"],
        bins=[-1, 0, 2, 5, 999],
        labels=["none", "low", "moderate", "high"]
    )

    # Visit type ratios (proportion of emergency and inpatient among total)
    df["visit_ratio_emergency"] = np.where(
        df["total_prior_visits"] > 0,
        df["number_emergency"] / df["total_prior_visits"],
        0
    )
    df["visit_ratio_inpatient"] = np.where(
        df["total_prior_visits"] > 0,
        df["number_inpatient"] / df["total_prior_visits"],
        0
    )

    # Number of medications bin
    df["num_medications_bin"] = pd.cut(
        df["num_medications"],
        bins=[0, 5, 10, 20, 81],
        labels=["low", "moderate", "high", "very_high"],
        right=True
    )

    # High utilizer flag (3+ prior visits in the past year)
    df["high_utilizer"] = (df["total_prior_visits"] >= 3).astype(int)

    # Procedure intensity (procedures per day of stay)
    df["procedure_intensity"] = np.where(
        df["time_in_hospital"] > 0,
        df["num_procedures"] / df["time_in_hospital"],
        0
    )

    # Number of diagnoses interaction with prior visits
    df["diagnoses_x_visits"] = df["number_diagnoses"] * df["total_prior_visits"]

    # Lab procedures per day
    df["lab_per_day"] = np.where(
        df["time_in_hospital"] > 0,
        df["num_lab_procedures"] / df["time_in_hospital"],
        0
    )

    new_features = [
        "los_bin", "total_prior_visits", "prior_visit_intensity",
        "visit_ratio_emergency", "visit_ratio_inpatient",
        "num_medications_bin", "high_utilizer", "procedure_intensity",
        "diagnoses_x_visits", "lab_per_day"
    ]
    print(f"\n  Created {len(new_features)} new features: {new_features}")
    return df


def handle_missing_values(df):
    """Handle missing values in the dataset."""
    print("\n--- Missing Value Handling ---")

    # Race: impute with mode
    if df["race"].isna().sum() > 0:
        mode_val = df["race"].mode()[0]
        n_missing = df["race"].isna().sum()
        df["race"] = df["race"].fillna(mode_val)
        print(f"  race: {n_missing} missing -> imputed with mode ('{mode_val}')")

    # Medical specialty: fill with 'Unknown'
    if "medical_specialty" in df.columns:
        n_missing = df["medical_specialty"].isna().sum()
        df["medical_specialty"] = df["medical_specialty"].fillna("Unknown")
        print(f"  medical_specialty: {n_missing} missing -> filled with 'Unknown'")

    # Diagnosis codes: fill missing with 'Unknown'
    for col in ["diag_1", "diag_2", "diag_3"]:
        if col in df.columns and df[col].isna().sum() > 0:
            n_missing = df[col].isna().sum()
            df[col] = df[col].fillna("Unknown")
            print(f"  {col}: {n_missing} missing -> filled with 'Unknown'")

    # Any remaining missing in categorical columns
    remaining = df.isna().sum()
    remaining = remaining[remaining > 0]
    if len(remaining) > 0:
        print(f"\n  Remaining missing values:")
        for col, count in remaining.items():
            if df[col].dtype == "object" or str(df[col].dtype) == "category":
                df[col] = df[col].fillna("Unknown")
            else:
                df[col] = df[col].fillna(df[col].median())
            print(f"    {col}: {count} -> imputed")

    print(f"  Total missing after handling: {df.isna().sum().sum()}")
    return df


def encode_categorical_features(df):
    """
    Encode categorical features for model training.

    - Binary medication columns: map to numeric
    - Ordinal features: label encode
    - High-cardinality categoricals: target encode or drop
    """
    print("\n--- Categorical Encoding ---")

    # Medication columns: map dosage changes to numeric
    med_columns = [
        "metformin", "repaglinide", "nateglinide", "chlorpropamide",
        "glimepiride", "acetohexamide", "glipizide", "glyburide",
        "tolbutamide", "pioglitazone", "rosiglitazone", "acarbose",
        "miglitol", "troglitazone", "tolazamide", "examide",
        "citoglipton", "insulin", "glyburide-metformin",
        "glipizide-metformin", "glimepiride-pioglitazone",
        "metformin-rosiglitazone", "metformin-pioglitazone"
    ]
    med_map = {"No": 0, "Steady": 1, "Down": 2, "Up": 3}

    for col in med_columns:
        if col in df.columns:
            df[col] = df[col].map(med_map).fillna(0).astype(int)

    # Binary columns
    binary_map = {"No": 0, "Yes": 1, "Ch": 1}
    for col in ["change", "diabetesMed"]:
        if col in df.columns:
            df[col] = df[col].map(binary_map).fillna(0).astype(int)

    # Age: convert to ordinal
    age_map = {
        "[0-10)": 0, "[10-20)": 1, "[20-30)": 2, "[30-40)": 3,
        "[40-50)": 4, "[50-60)": 5, "[60-70)": 6, "[70-80)": 7,
        "[80-90)": 8, "[90-100)": 9
    }
    if "age" in df.columns:
        df["age"] = df["age"].map(age_map).fillna(5).astype(int)

    # A1Cresult and max_glu_serum: ordinal encode
    a1c_map = {"None": 0, "Norm": 1, ">7": 2, ">8": 3}
    glu_map = {"None": 0, "Norm": 1, ">200": 2, ">300": 3}
    if "A1Cresult" in df.columns:
        df["A1Cresult"] = df["A1Cresult"].map(a1c_map).fillna(0).astype(int)
    if "max_glu_serum" in df.columns:
        df["max_glu_serum"] = df["max_glu_serum"].map(glu_map).fillna(0).astype(int)

    # Gender: binary encode
    if "gender" in df.columns:
        df["gender"] = df["gender"].map({"Female": 0, "Male": 1}).fillna(0).astype(int)

    # Drop high-cardinality string columns that would need complex encoding
    # (medical_specialty, diag_1, diag_2, diag_3 -> use grouped versions or drop)
    high_card_cols = ["medical_specialty", "diag_1", "diag_2", "diag_3"]
    for col in high_card_cols:
        if col in df.columns:
            # Create a simplified grouping for diagnosis codes
            if col.startswith("diag_"):
                df[col + "_group"] = df[col].apply(_group_diagnosis)
                df = df.drop(columns=[col])
            else:
                # Medical specialty: group into top categories
                top_specs = df[col].value_counts().head(10).index.tolist()
                df[col] = df[col].apply(
                    lambda x: x if x in top_specs else "Other"
                )

    # Label encode remaining object columns
    le = LabelEncoder()
    object_cols = df.select_dtypes(include=["object", "category"]).columns
    for col in object_cols:
        df[col] = df[col].astype(str)
        df[col] = le.fit_transform(df[col])
        print(f"  Label encoded: {col} ({df[col].nunique()} categories)")

    print(f"\n  Final encoded shape: {df.shape}")
    return df


def _group_diagnosis(code):
    """Group ICD-9 codes into broad categories."""
    if code == "Unknown" or pd.isna(code):
        return "Other"
    try:
        num = float(code)
    except (ValueError, TypeError):
        # E or V codes
        if str(code).startswith("E"):
            return "External_Causes"
        elif str(code).startswith("V"):
            return "Supplementary"
        return "Other"

    if 390 <= num <= 459 or num == 785:
        return "Circulatory"
    elif 460 <= num <= 519 or num == 786:
        return "Respiratory"
    elif 520 <= num <= 579 or num == 787:
        return "Digestive"
    elif 250 <= num < 251:
        return "Diabetes"
    elif 800 <= num <= 999:
        return "Injury"
    elif 710 <= num <= 739:
        return "Musculoskeletal"
    elif 580 <= num <= 629 or num == 788:
        return "Genitourinary"
    elif 140 <= num <= 239:
        return "Neoplasms"
    else:
        return "Other"


def scale_features(X_train, X_test):
    """Apply standard scaling to numeric features."""
    print("\n--- Feature Scaling ---")
    scaler = StandardScaler()
    feature_names = X_train.columns

    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train),
        columns=feature_names,
        index=X_train.index
    )
    X_test_scaled = pd.DataFrame(
        scaler.transform(X_test),
        columns=feature_names,
        index=X_test.index
    )

    print(f"  Scaled {len(feature_names)} features")
    return X_train_scaled, X_test_scaled, scaler


def main():
    """Run the full feature engineering pipeline."""
    print("=" * 60)
    print("FEATURE ENGINEERING PIPELINE")
    print("Diabetes 30-Day Readmission Prediction")
    print("=" * 60)

    # Load
    df = load_raw_data(RAW_DATA_PATH)

    # Clean
    df = clean_data(df)

    # Target variable
    df = create_target_variable(df)

    # Feature engineering
    df = engineer_time_based_features(df)

    # Missing values
    df = handle_missing_values(df)

    # Encode
    df = encode_categorical_features(df)

    # Split features and target
    target = "readmit_binary"
    X = df.drop(columns=[target])
    y = df[target]

    print(f"\n--- Final Dataset ---")
    print(f"  Features: {X.shape[1]}")
    print(f"  Samples: {X.shape[0]:,}")

    # Train/test split (stratified due to class imbalance)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )
    print(f"  Train set: {X_train.shape[0]:,}")
    print(f"  Test set: {X_test.shape[0]:,}")

    # Scale
    X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)

    # Save processed data
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    X_train_scaled.to_csv(os.path.join(OUTPUT_DIR, "X_train.csv"), index=False)
    X_test_scaled.to_csv(os.path.join(OUTPUT_DIR, "X_test.csv"), index=False)
    y_train.to_csv(os.path.join(OUTPUT_DIR, "y_train.csv"), index=False)
    y_test.to_csv(os.path.join(OUTPUT_DIR, "y_test.csv"), index=False)

    # Also save unscaled for tree-based models (XGBoost doesn't need scaling)
    X_train.to_csv(os.path.join(OUTPUT_DIR, "X_train_unscaled.csv"), index=False)
    X_test.to_csv(os.path.join(OUTPUT_DIR, "X_test_unscaled.csv"), index=False)

    # Save feature names
    with open(os.path.join(OUTPUT_DIR, "feature_names.txt"), "w") as f:
        f.write("\n".join(X.columns.tolist()))

    print(f"\n  Saved processed data to {OUTPUT_DIR}/")
    print(f"  Files: X_train.csv, X_test.csv, y_train.csv, y_test.csv")
    print(f"         X_train_unscaled.csv, X_test_unscaled.csv")
    print(f"         feature_names.txt")
    print("\n" + "=" * 60)
    print("FEATURE ENGINEERING COMPLETE")
    print("=" * 60)


if __name__ == "__main__":
    main()

FEATURE ENGINEERING PIPELINE
Diabetes 30-Day Readmission Prediction
Loading raw data from data/diabetic_data.csv...


FileNotFoundError: [Errno 2] No such file or directory: 'data/diabetic_data.csv'

In [ ]:
"""
03_xgboost_model.py
====================
XGBoost Model for Diabetes 30-Day Readmission Prediction

This script handles:
- Loading preprocessed data from feature engineering pipeline
- XGBoost model training with hyperparameter tuning
- Handling class imbalance via scale_pos_weight
- Cross-validation evaluation (AUC-ROC, Precision, Recall, F1)
- Feature importance analysis and visualization
- Model export for deployment

Author: Jiahao Li
Team: Final Project - G4 (Abdullah, Jiahao, Feifan)
Course: MSDS 422 - Practical Machine Learning
Date: February 2026

Dependencies:
    pip install xgboost scikit-learn pandas numpy matplotlib seaborn
"""

import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

import xgboost as xgb
from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV,
    cross_val_predict,
)
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
)
import joblib

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 150
plt.rcParams["font.size"] = 11

# ============================================================
# Configuration
# ============================================================
PROCESSED_DIR = os.path.join("data", "processed")
OUTPUT_DIR = os.path.join("outputs", "xgboost")
PLOT_DIR = os.path.join(OUTPUT_DIR, "plots")
RANDOM_STATE = 42
N_FOLDS = 5


def load_processed_data():
    """Load preprocessed train/test data."""
    print("Loading preprocessed data...")

    # Use unscaled data for XGBoost (tree-based models don't need scaling)
    X_train = pd.read_csv(os.path.join(PROCESSED_DIR, "X_train_unscaled.csv"))
    X_test = pd.read_csv(os.path.join(PROCESSED_DIR, "X_test_unscaled.csv"))
    y_train = pd.read_csv(os.path.join(PROCESSED_DIR, "y_train.csv")).squeeze()
    y_test = pd.read_csv(os.path.join(PROCESSED_DIR, "y_test.csv")).squeeze()

    print(f"  Train: {X_train.shape[0]:,} samples, {X_train.shape[1]} features")
    print(f"  Test:  {X_test.shape[0]:,} samples")
    print(f"  Class balance (train): {y_train.mean():.3f} positive rate")

    return X_train, X_test, y_train, y_test


def compute_scale_pos_weight(y):
    """Calculate scale_pos_weight for class imbalance."""
    neg = (y == 0).sum()
    pos = (y == 1).sum()
    weight = neg / pos
    print(f"  scale_pos_weight: {weight:.2f} (neg={neg:,}, pos={pos:,})")
    return weight


def train_xgboost_baseline(X_train, y_train, scale_weight):
    """Train baseline XGBoost with reasonable defaults."""
    print("\n--- Baseline XGBoost Training ---")

    model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=3,
        scale_pos_weight=scale_weight,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        eval_metric="auc",
        use_label_encoder=False,
        n_jobs=-1,
    )

    model.fit(X_train, y_train, verbose=False)
    print("  Baseline model trained successfully.")
    return model


def tune_hyperparameters(X_train, y_train, scale_weight):
    """
    Perform grid search for XGBoost hyperparameter tuning.

    Tuned parameters:
    - max_depth: tree depth (controls complexity)
    - learning_rate: step size shrinkage
    - n_estimators: number of boosting rounds
    - min_child_weight: minimum sum of instance weight in a child
    """
    print("\n--- Hyperparameter Tuning (GridSearchCV) ---")

    param_grid = {
        "max_depth": [4, 6, 8],
        "learning_rate": [0.05, 0.1],
        "n_estimators": [200, 300, 500],
        "min_child_weight": [1, 3, 5],
    }

    base_model = xgb.XGBClassifier(
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_weight,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        eval_metric="auc",
        use_label_encoder=False,
        n_jobs=-1,
    )

    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    grid_search = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1,
        verbose=1,
        refit=True,
    )

    grid_search.fit(X_train, y_train)

    print(f"\n  Best AUC-ROC (CV): {grid_search.best_score_:.4f}")
    print(f"  Best parameters: {grid_search.best_params_}")

    return grid_search.best_estimator_, grid_search.best_params_


def cross_validate_model(model, X_train, y_train):
    """Perform stratified k-fold cross-validation."""
    print(f"\n--- {N_FOLDS}-Fold Cross-Validation ---")

    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    # Get cross-validated predictions
    y_pred_proba = cross_val_predict(
        model, X_train, y_train, cv=cv, method="predict_proba"
    )[:, 1]
    y_pred = (y_pred_proba >= 0.5).astype(int)

    # Metrics
    metrics = {
        "cv_auc_roc": roc_auc_score(y_train, y_pred_proba),
        "cv_accuracy": accuracy_score(y_train, y_pred),
        "cv_precision": precision_score(y_train, y_pred),
        "cv_recall": recall_score(y_train, y_pred),
        "cv_f1": f1_score(y_train, y_pred),
        "cv_avg_precision": average_precision_score(y_train, y_pred_proba),
    }

    print(f"  AUC-ROC:           {metrics['cv_auc_roc']:.4f}")
    print(f"  Accuracy:          {metrics['cv_accuracy']:.4f}")
    print(f"  Precision:         {metrics['cv_precision']:.4f}")
    print(f"  Recall:            {metrics['cv_recall']:.4f}")
    print(f"  F1 Score:          {metrics['cv_f1']:.4f}")
    print(f"  Avg Precision (PR):{metrics['cv_avg_precision']:.4f}")

    return metrics


def evaluate_on_test(model, X_test, y_test):
    """Evaluate final model on hold-out test set."""
    print("\n--- Test Set Evaluation ---")

    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)

    metrics = {
        "test_auc_roc": roc_auc_score(y_test, y_pred_proba),
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_precision": precision_score(y_test, y_pred),
        "test_recall": recall_score(y_test, y_pred),
        "test_f1": f1_score(y_test, y_pred),
        "test_avg_precision": average_precision_score(y_test, y_pred_proba),
    }

    print(f"  AUC-ROC:           {metrics['test_auc_roc']:.4f}")
    print(f"  Accuracy:          {metrics['test_accuracy']:.4f}")
    print(f"  Precision:         {metrics['test_precision']:.4f}")
    print(f"  Recall:            {metrics['test_recall']:.4f}")
    print(f"  F1 Score:          {metrics['test_f1']:.4f}")
    print(f"  Avg Precision (PR):{metrics['test_avg_precision']:.4f}")

    print(f"\n  Classification Report:")
    print(classification_report(y_test, y_pred, target_names=["No Readmit", "Readmit <30d"]))

    return metrics, y_pred, y_pred_proba


# ============================================================
# Visualization Functions
# ============================================================

def plot_feature_importance(model, feature_names, top_n=20):
    """
    Plot top-N feature importances from XGBoost.

    Creates three importance plots:
    1. Gain-based importance (default)
    2. Weight-based (frequency) importance
    3. Cover-based importance
    """
    print(f"\n--- Feature Importance Plots (Top {top_n}) ---")
    os.makedirs(PLOT_DIR, exist_ok=True)

    # 1. Gain-based importance (most informative)
    importances = model.feature_importances_
    feat_imp = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances,
    }).sort_values("Importance", ascending=False).head(top_n)

    fig, ax = plt.subplots(figsize=(10, 8))
    bars = ax.barh(
        range(len(feat_imp)),
        feat_imp["Importance"].values,
        color=sns.color_palette("viridis", len(feat_imp)),
    )
    ax.set_yticks(range(len(feat_imp)))
    ax.set_yticklabels(feat_imp["Feature"].values)
    ax.invert_yaxis()
    ax.set_xlabel("Importance (Gain)")
    ax.set_title("XGBoost Feature Importance — Top 20 Features (Gain)")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Add value labels
    for bar, val in zip(bars, feat_imp["Importance"].values):
        ax.text(
            bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=9
        )

    plt.tight_layout()
    filepath = os.path.join(PLOT_DIR, "feature_importance_gain.png")
    plt.savefig(filepath, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {filepath}")

    # 2. XGBoost built-in importance plot (weight / frequency)
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))

    for idx, imp_type in enumerate(["weight", "cover"]):
        booster = model.get_booster()
        scores = booster.get_score(importance_type=imp_type)

        if scores:
            imp_df = (
                pd.DataFrame.from_dict(scores, orient="index", columns=["Importance"])
                .sort_values("Importance", ascending=False)
                .head(top_n)
            )
            # Map feature indices to names
            imp_df.index = [
                feature_names[int(f.replace("f", ""))]
                if f.startswith("f") and f[1:].isdigit()
                else f
                for f in imp_df.index
            ]

            axes[idx].barh(
                range(len(imp_df)),
                imp_df["Importance"].values,
                color=sns.color_palette("magma", len(imp_df))
                if idx == 0
                else sns.color_palette("cividis", len(imp_df)),
            )
            axes[idx].set_yticks(range(len(imp_df)))
            axes[idx].set_yticklabels(imp_df.index)
            axes[idx].invert_yaxis()
            axes[idx].set_xlabel(f"Importance ({imp_type.title()})")
            axes[idx].set_title(f"Feature Importance — {imp_type.title()}")
            axes[idx].spines["top"].set_visible(False)
            axes[idx].spines["right"].set_visible(False)

    plt.tight_layout()
    filepath = os.path.join(PLOT_DIR, "feature_importance_weight_cover.png")
    plt.savefig(filepath, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {filepath}")

    # Save importance data to CSV
    feat_imp_full = pd.DataFrame({
        "Feature": feature_names,
        "Importance_Gain": importances,
    }).sort_values("Importance_Gain", ascending=False)
    csv_path = os.path.join(OUTPUT_DIR, "feature_importance.csv")
    feat_imp_full.to_csv(csv_path, index=False)
    print(f"  Saved: {csv_path}")

    return feat_imp


def plot_roc_curve(y_test, y_pred_proba):
    """Plot ROC curve with AUC score."""
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    auc = roc_auc_score(y_test, y_pred_proba)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(fpr, tpr, color="#2563eb", linewidth=2, label=f"XGBoost (AUC = {auc:.4f})")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random (AUC = 0.5)")
    ax.fill_between(fpr, tpr, alpha=0.1, color="#2563eb")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curve — XGBoost Readmission Prediction")
    ax.legend(loc="lower right")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    filepath = os.path.join(PLOT_DIR, "roc_curve.png")
    plt.savefig(filepath, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {filepath}")


def plot_precision_recall_curve(y_test, y_pred_proba):
    """Plot Precision-Recall curve."""
    precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
    avg_prec = average_precision_score(y_test, y_pred_proba)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(recall, precision, color="#dc2626", linewidth=2,
            label=f"XGBoost (AP = {avg_prec:.4f})")
    ax.axhline(y=y_test.mean(), color="gray", linestyle="--", alpha=0.5,
               label=f"Baseline ({y_test.mean():.3f})")
    ax.fill_between(recall, precision, alpha=0.1, color="#dc2626")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Precision-Recall Curve — XGBoost")
    ax.legend()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    filepath = os.path.join(PLOT_DIR, "precision_recall_curve.png")
    plt.savefig(filepath, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {filepath}")


def plot_confusion_matrix(y_test, y_pred):
    """Plot confusion matrix heatmap."""
    cm = confusion_matrix(y_test, y_pred)

    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(
        cm, annot=True, fmt=",d", cmap="Blues",
        xticklabels=["No Readmit", "Readmit <30d"],
        yticklabels=["No Readmit", "Readmit <30d"],
        ax=ax
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title("Confusion Matrix — XGBoost")

    plt.tight_layout()
    filepath = os.path.join(PLOT_DIR, "confusion_matrix.png")
    plt.savefig(filepath, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {filepath}")


def plot_learning_curve(model, X_train, y_train):
    """Plot XGBoost training history with eval metric."""
    print("\n--- Learning Curve ---")

    eval_set = [(X_train, y_train)]
    model_lc = xgb.XGBClassifier(**model.get_params())
    model_lc.set_params(eval_metric="logloss")

    model_lc.fit(
        X_train, y_train,
        eval_set=eval_set,
        verbose=False,
    )

    results = model_lc.evals_result()
    epochs = range(len(results["validation_0"]["logloss"]))

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, results["validation_0"]["logloss"], color="#2563eb", linewidth=1.5)
    ax.set_xlabel("Boosting Rounds")
    ax.set_ylabel("Log Loss")
    ax.set_title("XGBoost Training Loss Curve")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    filepath = os.path.join(PLOT_DIR, "learning_curve.png")
    plt.savefig(filepath, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {filepath}")


# ============================================================
# Main
# ============================================================

def main():
    """Run the full XGBoost pipeline."""
    print("=" * 60)
    print("XGBOOST MODEL PIPELINE")
    print("Diabetes 30-Day Readmission Prediction")
    print(f"Run Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    print("=" * 60)

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(PLOT_DIR, exist_ok=True)

    # Load data
    X_train, X_test, y_train, y_test = load_processed_data()
    feature_names = X_train.columns.tolist()

    # Class imbalance weight
    scale_weight = compute_scale_pos_weight(y_train)

    # Train baseline
    baseline_model = train_xgboost_baseline(X_train, y_train, scale_weight)

    # Hyperparameter tuning
    best_model, best_params = tune_hyperparameters(X_train, y_train, scale_weight)

    # Cross-validation
    cv_metrics = cross_validate_model(best_model, X_train, y_train)

    # Test evaluation
    test_metrics, y_pred, y_pred_proba = evaluate_on_test(best_model, X_test, y_test)

    # ---- Visualizations ----
    print("\n" + "=" * 60)
    print("GENERATING VISUALIZATIONS")
    print("=" * 60)

    plot_feature_importance(best_model, feature_names, top_n=20)
    plot_roc_curve(y_test, y_pred_proba)
    plot_precision_recall_curve(y_test, y_pred_proba)
    plot_confusion_matrix(y_test, y_pred)
    plot_learning_curve(best_model, X_train, y_train)

    # ---- Save Model & Results ----
    print("\n" + "=" * 60)
    print("SAVING MODEL & RESULTS")
    print("=" * 60)

    # Save model
    model_path = os.path.join(OUTPUT_DIR, "xgboost_best_model.joblib")
    joblib.dump(best_model, model_path)
    print(f"  Model saved: {model_path}")

    # Save all metrics
    all_metrics = {**cv_metrics, **test_metrics, "best_params": best_params}
    metrics_path = os.path.join(OUTPUT_DIR, "metrics.json")
    with open(metrics_path, "w") as f:
        json.dump(all_metrics, f, indent=2, default=str)
    print(f"  Metrics saved: {metrics_path}")

    # Summary
    print("\n" + "=" * 60)
    print("XGBOOST PIPELINE COMPLETE")
    print("=" * 60)
    print(f"\n  Best Parameters: {best_params}")
    print(f"  CV AUC-ROC:   {cv_metrics['cv_auc_roc']:.4f}")
    print(f"  Test AUC-ROC: {test_metrics['test_auc_roc']:.4f}")
    print(f"  Test F1:      {test_metrics['test_f1']:.4f}")
    print(f"\n  All outputs saved to: {OUTPUT_DIR}/")
    print(f"  Plots saved to:       {PLOT_DIR}/")


if __name__ == "__main__":
    main()

In [ ]:
"""
03a_logistic_regression.py
===========================
Logistic Regression Model for Diabetes 30-Day Readmission Prediction

Baseline interpretable model with L2 regularization and threshold tuning.

Author: Abdullah Abdulsami
Team: Final Project - G4 (Abdullah, Jiahao, Feifan)
Course: MSDS 422 - Practical Machine Learning
Date: February 2026
"""

import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    accuracy_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score,
)
import joblib

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 150
plt.rcParams["font.size"] = 11

# ============================================================
# Configuration
# ============================================================
PROCESSED_DIR = os.path.join("data", "processed")
OUTPUT_DIR = os.path.join("outputs", "logistic_regression")
PLOT_DIR = os.path.join(OUTPUT_DIR, "plots")
RANDOM_STATE = 42
N_FOLDS = 5


def load_processed_data():
    """Load preprocessed train/test data (scaled for LR)."""
    print("Loading preprocessed data (scaled)...")
    X_train = pd.read_csv(os.path.join(PROCESSED_DIR, "X_train.csv"))
    X_test = pd.read_csv(os.path.join(PROCESSED_DIR, "X_test.csv"))
    y_train = pd.read_csv(os.path.join(PROCESSED_DIR, "y_train.csv")).squeeze()
    y_test = pd.read_csv(os.path.join(PROCESSED_DIR, "y_test.csv")).squeeze()

    print(f"  Train: {X_train.shape[0]:,} samples, {X_train.shape[1]} features")
    print(f"  Test:  {X_test.shape[0]:,} samples")
    print(f"  Positive rate (train): {y_train.mean():.3f}")
    return X_train, X_test, y_train, y_test


def tune_and_train(X_train, y_train):
    """Hyperparameter tuning with GridSearchCV."""
    print("\n--- Hyperparameter Tuning (GridSearchCV) ---")

    param_grid = {
        "C": [0.001, 0.01, 0.1, 1.0, 10.0],
        "penalty": ["l2"],
        "solver": ["lbfgs"],
        "class_weight": ["balanced", None],
    }

    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    grid = GridSearchCV(
        LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        param_grid, cv=cv, scoring="roc_auc", n_jobs=-1, verbose=1, refit=True,
    )
    grid.fit(X_train, y_train)

    print(f"\n  Best AUC-ROC (CV): {grid.best_score_:.4f}")
    print(f"  Best parameters: {grid.best_params_}")
    return grid.best_estimator_, grid.best_params_


def cross_validate_model(model, X_train, y_train):
    """5-fold cross-validation metrics."""
    print(f"\n--- {N_FOLDS}-Fold Cross-Validation ---")
    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    y_proba = cross_val_predict(model, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)

    metrics = {
        "cv_auc_roc": roc_auc_score(y_train, y_proba),
        "cv_accuracy": accuracy_score(y_train, y_pred),
        "cv_precision": precision_score(y_train, y_pred),
        "cv_recall": recall_score(y_train, y_pred),
        "cv_f1": f1_score(y_train, y_pred),
        "cv_avg_precision": average_precision_score(y_train, y_proba),
    }

    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")
    return metrics


def evaluate_on_test(model, X_test, y_test):
    """Evaluate on hold-out test set."""
    print("\n--- Test Set Evaluation ---")
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)

    metrics = {
        "test_auc_roc": roc_auc_score(y_test, y_proba),
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_precision": precision_score(y_test, y_pred),
        "test_recall": recall_score(y_test, y_pred),
        "test_f1": f1_score(y_test, y_pred),
        "test_avg_precision": average_precision_score(y_test, y_proba),
    }

    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

    print(f"\n  Classification Report:")
    print(classification_report(y_test, y_pred,
          target_names=["No Readmit", "Readmit <30d"]))

    return metrics, y_pred, y_proba


def plot_coefficient_importance(model, feature_names, top_n=20):
    """Plot top feature coefficients (interpretability)."""
    os.makedirs(PLOT_DIR, exist_ok=True)

    coefs = model.coef_[0]
    coef_df = pd.DataFrame({
        "Feature": feature_names, "Coefficient": coefs,
        "Abs_Coef": np.abs(coefs)
    }).sort_values("Abs_Coef", ascending=False).head(top_n)

    fig, ax = plt.subplots(figsize=(10, 8))
    colors = ["#ef4444" if c < 0 else "#22c55e" for c in coef_df["Coefficient"].values]
    ax.barh(range(len(coef_df)), coef_df["Coefficient"].values, color=colors)
    ax.set_yticks(range(len(coef_df)))
    ax.set_yticklabels(coef_df["Feature"].values)
    ax.invert_yaxis()
    ax.set_xlabel("Coefficient Value")
    ax.set_title("Logistic Regression — Top 20 Feature Coefficients")
    ax.axvline(x=0, color="black", linewidth=0.5)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    path = os.path.join(PLOT_DIR, "lr_coefficients.png")
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")


def plot_roc_curve(y_test, y_proba):
    """Plot ROC curve."""
    os.makedirs(PLOT_DIR, exist_ok=True)
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(fpr, tpr, color="#8b5cf6", linewidth=2, label=f"Logistic Reg (AUC = {auc:.4f})")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.5)
    ax.fill_between(fpr, tpr, alpha=0.1, color="#8b5cf6")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curve — Logistic Regression")
    ax.legend(loc="lower right")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    path = os.path.join(PLOT_DIR, "roc_curve.png")
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")


def plot_confusion_matrix(y_test, y_pred):
    """Plot confusion matrix."""
    os.makedirs(PLOT_DIR, exist_ok=True)
    cm = confusion_matrix(y_test, y_pred)

    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt=",d", cmap="Purples",
                xticklabels=["No Readmit", "Readmit <30d"],
                yticklabels=["No Readmit", "Readmit <30d"], ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title("Confusion Matrix — Logistic Regression")

    plt.tight_layout()
    path = os.path.join(PLOT_DIR, "confusion_matrix.png")
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")


def main():
    print("=" * 60)
    print("LOGISTIC REGRESSION MODEL PIPELINE")
    print("Diabetes 30-Day Readmission Prediction")
    print("=" * 60)

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(PLOT_DIR, exist_ok=True)

    X_train, X_test, y_train, y_test = load_processed_data()
    feature_names = X_train.columns.tolist()

    best_model, best_params = tune_and_train(X_train, y_train)
    cv_metrics = cross_validate_model(best_model, X_train, y_train)
    test_metrics, y_pred, y_proba = evaluate_on_test(best_model, X_test, y_test)

    # Plots
    plot_coefficient_importance(best_model, feature_names)
    plot_roc_curve(y_test, y_proba)
    plot_confusion_matrix(y_test, y_pred)

    # Save
    joblib.dump(best_model, os.path.join(OUTPUT_DIR, "logistic_regression_model.joblib"))
    all_metrics = {**cv_metrics, **test_metrics, "best_params": best_params}
    with open(os.path.join(OUTPUT_DIR, "metrics.json"), "w") as f:
        json.dump(all_metrics, f, indent=2, default=str)

    print(f"\n{'=' * 60}")
    print("LOGISTIC REGRESSION COMPLETE")
    print(f"  CV AUC-ROC:   {cv_metrics['cv_auc_roc']:.4f}")
    print(f"  Test AUC-ROC: {test_metrics['test_auc_roc']:.4f}")
    print(f"  Test F1:      {test_metrics['test_f1']:.4f}")
    print(f"  Outputs: {OUTPUT_DIR}/")
    print("=" * 60)


if __name__ == "__main__":
    main()

In [ ]:
"""
03b_random_forest.py
=====================
Random Forest Model for Diabetes 30-Day Readmission Prediction

Ensemble model with built-in feature importance and class balancing.

Author: Abdullah Abdulsami
Team: Final Project - G4 (Abdullah, Jiahao, Feifan)
Course: MSDS 422 - Practical Machine Learning
Date: February 2026
"""

import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    accuracy_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score,
)
import joblib

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 150
plt.rcParams["font.size"] = 11

# ============================================================
# Configuration
# ============================================================
PROCESSED_DIR = os.path.join("data", "processed")
OUTPUT_DIR = os.path.join("outputs", "random_forest")
PLOT_DIR = os.path.join(OUTPUT_DIR, "plots")
RANDOM_STATE = 42
N_FOLDS = 5


def load_processed_data():
    """Load preprocessed data (unscaled — RF doesn't need scaling)."""
    print("Loading preprocessed data (unscaled)...")
    X_train = pd.read_csv(os.path.join(PROCESSED_DIR, "X_train_unscaled.csv"))
    X_test = pd.read_csv(os.path.join(PROCESSED_DIR, "X_test_unscaled.csv"))
    y_train = pd.read_csv(os.path.join(PROCESSED_DIR, "y_train.csv")).squeeze()
    y_test = pd.read_csv(os.path.join(PROCESSED_DIR, "y_test.csv")).squeeze()

    print(f"  Train: {X_train.shape[0]:,} samples, {X_train.shape[1]} features")
    print(f"  Test:  {X_test.shape[0]:,} samples")
    print(f"  Positive rate (train): {y_train.mean():.3f}")
    return X_train, X_test, y_train, y_test


def tune_and_train(X_train, y_train):
    """Hyperparameter tuning for Random Forest."""
    print("\n--- Hyperparameter Tuning (GridSearchCV) ---")

    param_grid = {
        "n_estimators": [200, 300, 500],
        "max_depth": [10, 15, 20, None],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "class_weight": ["balanced", "balanced_subsample"],
    }

    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    # Use a smaller grid for speed — randomized search with key combos
    from sklearn.model_selection import RandomizedSearchCV

    grid = RandomizedSearchCV(
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        param_grid, n_iter=30, cv=cv, scoring="roc_auc",
        n_jobs=-1, verbose=1, refit=True, random_state=RANDOM_STATE,
    )
    grid.fit(X_train, y_train)

    print(f"\n  Best AUC-ROC (CV): {grid.best_score_:.4f}")
    print(f"  Best parameters: {grid.best_params_}")
    return grid.best_estimator_, grid.best_params_


def cross_validate_model(model, X_train, y_train):
    """5-fold cross-validation metrics."""
    print(f"\n--- {N_FOLDS}-Fold Cross-Validation ---")
    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    y_proba = cross_val_predict(model, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)

    metrics = {
        "cv_auc_roc": roc_auc_score(y_train, y_proba),
        "cv_accuracy": accuracy_score(y_train, y_pred),
        "cv_precision": precision_score(y_train, y_pred),
        "cv_recall": recall_score(y_train, y_pred),
        "cv_f1": f1_score(y_train, y_pred),
        "cv_avg_precision": average_precision_score(y_train, y_proba),
    }

    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")
    return metrics


def evaluate_on_test(model, X_test, y_test):
    """Evaluate on hold-out test set."""
    print("\n--- Test Set Evaluation ---")
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)

    metrics = {
        "test_auc_roc": roc_auc_score(y_test, y_proba),
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_precision": precision_score(y_test, y_pred),
        "test_recall": recall_score(y_test, y_pred),
        "test_f1": f1_score(y_test, y_pred),
        "test_avg_precision": average_precision_score(y_test, y_proba),
    }

    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

    print(f"\n  Classification Report:")
    print(classification_report(y_test, y_pred,
          target_names=["No Readmit", "Readmit <30d"]))

    return metrics, y_pred, y_proba


def plot_feature_importance(model, feature_names, top_n=20):
    """Plot Random Forest feature importance (Gini impurity)."""
    os.makedirs(PLOT_DIR, exist_ok=True)

    importances = model.feature_importances_
    std = np.std([tree.feature_importances_ for tree in model.estimators_], axis=0)

    feat_imp = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances,
        "Std": std,
    }).sort_values("Importance", ascending=False).head(top_n)

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(
        range(len(feat_imp)), feat_imp["Importance"].values,
        xerr=feat_imp["Std"].values,
        color=sns.color_palette("YlOrRd_r", len(feat_imp)),
        capsize=3,
    )
    ax.set_yticks(range(len(feat_imp)))
    ax.set_yticklabels(feat_imp["Feature"].values)
    ax.invert_yaxis()
    ax.set_xlabel("Importance (Gini)")
    ax.set_title("Random Forest Feature Importance — Top 20 (with Std Dev)")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    path = os.path.join(PLOT_DIR, "rf_feature_importance.png")
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")

    # Save CSV
    feat_imp_full = pd.DataFrame({
        "Feature": feature_names, "Importance": importances,
    }).sort_values("Importance", ascending=False)
    feat_imp_full.to_csv(os.path.join(OUTPUT_DIR, "feature_importance.csv"), index=False)


def plot_roc_curve(y_test, y_proba):
    """Plot ROC curve."""
    os.makedirs(PLOT_DIR, exist_ok=True)
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(fpr, tpr, color="#f59e0b", linewidth=2, label=f"Random Forest (AUC = {auc:.4f})")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.5)
    ax.fill_between(fpr, tpr, alpha=0.1, color="#f59e0b")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curve — Random Forest")
    ax.legend(loc="lower right")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    path = os.path.join(PLOT_DIR, "roc_curve.png")
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")


def plot_confusion_matrix(y_test, y_pred):
    """Plot confusion matrix."""
    os.makedirs(PLOT_DIR, exist_ok=True)
    cm = confusion_matrix(y_test, y_pred)

    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt=",d", cmap="YlOrBr",
                xticklabels=["No Readmit", "Readmit <30d"],
                yticklabels=["No Readmit", "Readmit <30d"], ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title("Confusion Matrix — Random Forest")

    plt.tight_layout()
    path = os.path.join(PLOT_DIR, "confusion_matrix.png")
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")


def main():
    print("=" * 60)
    print("RANDOM FOREST MODEL PIPELINE")
    print("Diabetes 30-Day Readmission Prediction")
    print("=" * 60)

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(PLOT_DIR, exist_ok=True)

    X_train, X_test, y_train, y_test = load_processed_data()
    feature_names = X_train.columns.tolist()

    best_model, best_params = tune_and_train(X_train, y_train)
    cv_metrics = cross_validate_model(best_model, X_train, y_train)
    test_metrics, y_pred, y_proba = evaluate_on_test(best_model, X_test, y_test)

    # Plots
    plot_feature_importance(best_model, feature_names)
    plot_roc_curve(y_test, y_proba)
    plot_confusion_matrix(y_test, y_pred)

    # Save
    joblib.dump(best_model, os.path.join(OUTPUT_DIR, "random_forest_model.joblib"))
    all_metrics = {**cv_metrics, **test_metrics, "best_params": best_params}
    with open(os.path.join(OUTPUT_DIR, "metrics.json"), "w") as f:
        json.dump(all_metrics, f, indent=2, default=str)

    print(f"\n{'=' * 60}")
    print("RANDOM FOREST COMPLETE")
    print(f"  CV AUC-ROC:   {cv_metrics['cv_auc_roc']:.4f}")
    print(f"  Test AUC-ROC: {test_metrics['test_auc_roc']:.4f}")
    print(f"  Test F1:      {test_metrics['test_f1']:.4f}")
    print(f"  Outputs: {OUTPUT_DIR}/")
    print("=" * 60)


if __name__ == "__main__":
    main()

In [ ]:
"""
03_xgboost_model.py
====================
XGBoost Model for Diabetes 30-Day Readmission Prediction

This script handles:
- Loading preprocessed data from feature engineering pipeline
- XGBoost model training with hyperparameter tuning
- Handling class imbalance via scale_pos_weight
- Cross-validation evaluation (AUC-ROC, Precision, Recall, F1)
- Feature importance analysis and visualization
- Model export for deployment

Author: Jiahao Li
Team: Final Project - G4 (Abdullah, Jiahao, Feifan)
Course: MSDS 422 - Practical Machine Learning
Date: February 2026

Dependencies:
    pip install xgboost scikit-learn pandas numpy matplotlib seaborn
"""

import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

import xgboost as xgb
from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV,
    cross_val_predict,
)
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
)
import joblib

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 150
plt.rcParams["font.size"] = 11

# ============================================================
# Configuration
# ============================================================
PROCESSED_DIR = os.path.join("data", "processed")
OUTPUT_DIR = os.path.join("outputs", "xgboost")
PLOT_DIR = os.path.join(OUTPUT_DIR, "plots")
RANDOM_STATE = 42
N_FOLDS = 5


def load_processed_data():
    """Load preprocessed train/test data."""
    print("Loading preprocessed data...")

    # Use unscaled data for XGBoost (tree-based models don't need scaling)
    X_train = pd.read_csv(os.path.join(PROCESSED_DIR, "X_train_unscaled.csv"))
    X_test = pd.read_csv(os.path.join(PROCESSED_DIR, "X_test_unscaled.csv"))
    y_train = pd.read_csv(os.path.join(PROCESSED_DIR, "y_train.csv")).squeeze()
    y_test = pd.read_csv(os.path.join(PROCESSED_DIR, "y_test.csv")).squeeze()

    print(f"  Train: {X_train.shape[0]:,} samples, {X_train.shape[1]} features")
    print(f"  Test:  {X_test.shape[0]:,} samples")
    print(f"  Class balance (train): {y_train.mean():.3f} positive rate")

    return X_train, X_test, y_train, y_test


def compute_scale_pos_weight(y):
    """Calculate scale_pos_weight for class imbalance."""
    neg = (y == 0).sum()
    pos = (y == 1).sum()
    weight = neg / pos
    print(f"  scale_pos_weight: {weight:.2f} (neg={neg:,}, pos={pos:,})")
    return weight


def train_xgboost_baseline(X_train, y_train, scale_weight):
    """Train baseline XGBoost with reasonable defaults."""
    print("\n--- Baseline XGBoost Training ---")

    model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=3,
        scale_pos_weight=scale_weight,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        eval_metric="auc",
        use_label_encoder=False,
        n_jobs=-1,
    )

    model.fit(X_train, y_train, verbose=False)
    print("  Baseline model trained successfully.")
    return model


def tune_hyperparameters(X_train, y_train, scale_weight):
    """
    Perform grid search for XGBoost hyperparameter tuning.

    Tuned parameters:
    - max_depth: tree depth (controls complexity)
    - learning_rate: step size shrinkage
    - n_estimators: number of boosting rounds
    - min_child_weight: minimum sum of instance weight in a child
    """
    print("\n--- Hyperparameter Tuning (GridSearchCV) ---")

    param_grid = {
        "max_depth": [4, 6, 8],
        "learning_rate": [0.05, 0.1],
        "n_estimators": [200, 300, 500],
        "min_child_weight": [1, 3, 5],
    }

    base_model = xgb.XGBClassifier(
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_weight,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        eval_metric="auc",
        use_label_encoder=False,
        n_jobs=-1,
    )

    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    grid_search = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1,
        verbose=1,
        refit=True,
    )

    grid_search.fit(X_train, y_train)

    print(f"\n  Best AUC-ROC (CV): {grid_search.best_score_:.4f}")
    print(f"  Best parameters: {grid_search.best_params_}")

    return grid_search.best_estimator_, grid_search.best_params_


def cross_validate_model(model, X_train, y_train):
    """Perform stratified k-fold cross-validation."""
    print(f"\n--- {N_FOLDS}-Fold Cross-Validation ---")

    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    # Get cross-validated predictions
    y_pred_proba = cross_val_predict(
        model, X_train, y_train, cv=cv, method="predict_proba"
    )[:, 1]
    y_pred = (y_pred_proba >= 0.5).astype(int)

    # Metrics
    metrics = {
        "cv_auc_roc": roc_auc_score(y_train, y_pred_proba),
        "cv_accuracy": accuracy_score(y_train, y_pred),
        "cv_precision": precision_score(y_train, y_pred),
        "cv_recall": recall_score(y_train, y_pred),
        "cv_f1": f1_score(y_train, y_pred),
        "cv_avg_precision": average_precision_score(y_train, y_pred_proba),
    }

    print(f"  AUC-ROC:           {metrics['cv_auc_roc']:.4f}")
    print(f"  Accuracy:          {metrics['cv_accuracy']:.4f}")
    print(f"  Precision:         {metrics['cv_precision']:.4f}")
    print(f"  Recall:            {metrics['cv_recall']:.4f}")
    print(f"  F1 Score:          {metrics['cv_f1']:.4f}")
    print(f"  Avg Precision (PR):{metrics['cv_avg_precision']:.4f}")

    return metrics


def evaluate_on_test(model, X_test, y_test):
    """Evaluate final model on hold-out test set."""
    print("\n--- Test Set Evaluation ---")

    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)

    metrics = {
        "test_auc_roc": roc_auc_score(y_test, y_pred_proba),
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_precision": precision_score(y_test, y_pred),
        "test_recall": recall_score(y_test, y_pred),
        "test_f1": f1_score(y_test, y_pred),
        "test_avg_precision": average_precision_score(y_test, y_pred_proba),
    }

    print(f"  AUC-ROC:           {metrics['test_auc_roc']:.4f}")
    print(f"  Accuracy:          {metrics['test_accuracy']:.4f}")
    print(f"  Precision:         {metrics['test_precision']:.4f}")
    print(f"  Recall:            {metrics['test_recall']:.4f}")
    print(f"  F1 Score:          {metrics['test_f1']:.4f}")
    print(f"  Avg Precision (PR):{metrics['test_avg_precision']:.4f}")

    print(f"\n  Classification Report:")
    print(classification_report(y_test, y_pred, target_names=["No Readmit", "Readmit <30d"]))

    return metrics, y_pred, y_pred_proba


# ============================================================
# Visualization Functions
# ============================================================

def plot_feature_importance(model, feature_names, top_n=20):
    """
    Plot top-N feature importances from XGBoost.

    Creates three importance plots:
    1. Gain-based importance (default)
    2. Weight-based (frequency) importance
    3. Cover-based importance
    """
    print(f"\n--- Feature Importance Plots (Top {top_n}) ---")
    os.makedirs(PLOT_DIR, exist_ok=True)

    # 1. Gain-based importance (most informative)
    importances = model.feature_importances_
    feat_imp = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances,
    }).sort_values("Importance", ascending=False).head(top_n)

    fig, ax = plt.subplots(figsize=(10, 8))
    bars = ax.barh(
        range(len(feat_imp)),
        feat_imp["Importance"].values,
        color=sns.color_palette("viridis", len(feat_imp)),
    )
    ax.set_yticks(range(len(feat_imp)))
    ax.set_yticklabels(feat_imp["Feature"].values)
    ax.invert_yaxis()
    ax.set_xlabel("Importance (Gain)")
    ax.set_title("XGBoost Feature Importance — Top 20 Features (Gain)")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Add value labels
    for bar, val in zip(bars, feat_imp["Importance"].values):
        ax.text(
            bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=9
        )

    plt.tight_layout()
    filepath = os.path.join(PLOT_DIR, "feature_importance_gain.png")
    plt.savefig(filepath, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {filepath}")

    # 2. XGBoost built-in importance plot (weight / frequency)
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))

    for idx, imp_type in enumerate(["weight", "cover"]):
        booster = model.get_booster()
        scores = booster.get_score(importance_type=imp_type)

        if scores:
            imp_df = (
                pd.DataFrame.from_dict(scores, orient="index", columns=["Importance"])
                .sort_values("Importance", ascending=False)
                .head(top_n)
            )
            # Map feature indices to names
            imp_df.index = [
                feature_names[int(f.replace("f", ""))]
                if f.startswith("f") and f[1:].isdigit()
                else f
                for f in imp_df.index
            ]

            axes[idx].barh(
                range(len(imp_df)),
                imp_df["Importance"].values,
                color=sns.color_palette("magma", len(imp_df))
                if idx == 0
                else sns.color_palette("cividis", len(imp_df)),
            )
            axes[idx].set_yticks(range(len(imp_df)))
            axes[idx].set_yticklabels(imp_df.index)
            axes[idx].invert_yaxis()
            axes[idx].set_xlabel(f"Importance ({imp_type.title()})")
            axes[idx].set_title(f"Feature Importance — {imp_type.title()}")
            axes[idx].spines["top"].set_visible(False)
            axes[idx].spines["right"].set_visible(False)

    plt.tight_layout()
    filepath = os.path.join(PLOT_DIR, "feature_importance_weight_cover.png")
    plt.savefig(filepath, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {filepath}")

    # Save importance data to CSV
    feat_imp_full = pd.DataFrame({
        "Feature": feature_names,
        "Importance_Gain": importances,
    }).sort_values("Importance_Gain", ascending=False)
    csv_path = os.path.join(OUTPUT_DIR, "feature_importance.csv")
    feat_imp_full.to_csv(csv_path, index=False)
    print(f"  Saved: {csv_path}")

    return feat_imp


def plot_roc_curve(y_test, y_pred_proba):
    """Plot ROC curve with AUC score."""
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    auc = roc_auc_score(y_test, y_pred_proba)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(fpr, tpr, color="#2563eb", linewidth=2, label=f"XGBoost (AUC = {auc:.4f})")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random (AUC = 0.5)")
    ax.fill_between(fpr, tpr, alpha=0.1, color="#2563eb")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curve — XGBoost Readmission Prediction")
    ax.legend(loc="lower right")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    filepath = os.path.join(PLOT_DIR, "roc_curve.png")
    plt.savefig(filepath, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {filepath}")


def plot_precision_recall_curve(y_test, y_pred_proba):
    """Plot Precision-Recall curve."""
    precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
    avg_prec = average_precision_score(y_test, y_pred_proba)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(recall, precision, color="#dc2626", linewidth=2,
            label=f"XGBoost (AP = {avg_prec:.4f})")
    ax.axhline(y=y_test.mean(), color="gray", linestyle="--", alpha=0.5,
               label=f"Baseline ({y_test.mean():.3f})")
    ax.fill_between(recall, precision, alpha=0.1, color="#dc2626")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Precision-Recall Curve — XGBoost")
    ax.legend()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    filepath = os.path.join(PLOT_DIR, "precision_recall_curve.png")
    plt.savefig(filepath, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {filepath}")


def plot_confusion_matrix(y_test, y_pred):
    """Plot confusion matrix heatmap."""
    cm = confusion_matrix(y_test, y_pred)

    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(
        cm, annot=True, fmt=",d", cmap="Blues",
        xticklabels=["No Readmit", "Readmit <30d"],
        yticklabels=["No Readmit", "Readmit <30d"],
        ax=ax
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title("Confusion Matrix — XGBoost")

    plt.tight_layout()
    filepath = os.path.join(PLOT_DIR, "confusion_matrix.png")
    plt.savefig(filepath, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {filepath}")


def plot_learning_curve(model, X_train, y_train):
    """Plot XGBoost training history with eval metric."""
    print("\n--- Learning Curve ---")

    eval_set = [(X_train, y_train)]
    model_lc = xgb.XGBClassifier(**model.get_params())
    model_lc.set_params(eval_metric="logloss")

    model_lc.fit(
        X_train, y_train,
        eval_set=eval_set,
        verbose=False,
    )

    results = model_lc.evals_result()
    epochs = range(len(results["validation_0"]["logloss"]))

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, results["validation_0"]["logloss"], color="#2563eb", linewidth=1.5)
    ax.set_xlabel("Boosting Rounds")
    ax.set_ylabel("Log Loss")
    ax.set_title("XGBoost Training Loss Curve")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    filepath = os.path.join(PLOT_DIR, "learning_curve.png")
    plt.savefig(filepath, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {filepath}")


# ============================================================
# Main
# ============================================================

def main():
    """Run the full XGBoost pipeline."""
    print("=" * 60)
    print("XGBOOST MODEL PIPELINE")
    print("Diabetes 30-Day Readmission Prediction")
    print(f"Run Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    print("=" * 60)

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(PLOT_DIR, exist_ok=True)

    # Load data
    X_train, X_test, y_train, y_test = load_processed_data()
    feature_names = X_train.columns.tolist()

    # Class imbalance weight
    scale_weight = compute_scale_pos_weight(y_train)

    # Train baseline
    baseline_model = train_xgboost_baseline(X_train, y_train, scale_weight)

    # Hyperparameter tuning
    best_model, best_params = tune_hyperparameters(X_train, y_train, scale_weight)

    # Cross-validation
    cv_metrics = cross_validate_model(best_model, X_train, y_train)

    # Test evaluation
    test_metrics, y_pred, y_pred_proba = evaluate_on_test(best_model, X_test, y_test)

    # ---- Visualizations ----
    print("\n" + "=" * 60)
    print("GENERATING VISUALIZATIONS")
    print("=" * 60)

    plot_feature_importance(best_model, feature_names, top_n=20)
    plot_roc_curve(y_test, y_pred_proba)
    plot_precision_recall_curve(y_test, y_pred_proba)
    plot_confusion_matrix(y_test, y_pred)
    plot_learning_curve(best_model, X_train, y_train)

    # ---- Save Model & Results ----
    print("\n" + "=" * 60)
    print("SAVING MODEL & RESULTS")
    print("=" * 60)

    # Save model
    model_path = os.path.join(OUTPUT_DIR, "xgboost_best_model.joblib")
    joblib.dump(best_model, model_path)
    print(f"  Model saved: {model_path}")

    # Save all metrics
    all_metrics = {**cv_metrics, **test_metrics, "best_params": best_params}
    metrics_path = os.path.join(OUTPUT_DIR, "metrics.json")
    with open(metrics_path, "w") as f:
        json.dump(all_metrics, f, indent=2, default=str)
    print(f"  Metrics saved: {metrics_path}")

    # Summary
    print("\n" + "=" * 60)
    print("XGBOOST PIPELINE COMPLETE")
    print("=" * 60)
    print(f"\n  Best Parameters: {best_params}")
    print(f"  CV AUC-ROC:   {cv_metrics['cv_auc_roc']:.4f}")
    print(f"  Test AUC-ROC: {test_metrics['test_auc_roc']:.4f}")
    print(f"  Test F1:      {test_metrics['test_f1']:.4f}")
    print(f"\n  All outputs saved to: {OUTPUT_DIR}/")
    print(f"  Plots saved to:       {PLOT_DIR}/")


if __name__ == "__main__":
    main()

In [ ]:
"""
03d_mlp_neural_network.py
==========================
MLP Neural Network Model for Diabetes 30-Day Readmission Prediction

Deep learning approach using sklearn MLPClassifier with early stopping.

Author: Feifan
Team: Final Project - G4 (Abdullah, Jiahao, Feifan)
Course: MSDS 422 - Practical Machine Learning
Date: February 2026
"""

import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    accuracy_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score,
)
import joblib

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 150
plt.rcParams["font.size"] = 11

# ============================================================
# Configuration
# ============================================================
PROCESSED_DIR = os.path.join("data", "processed")
OUTPUT_DIR = os.path.join("outputs", "mlp")
PLOT_DIR = os.path.join(OUTPUT_DIR, "plots")
RANDOM_STATE = 42
N_FOLDS = 5


def load_processed_data():
    """Load preprocessed data (scaled — required for MLP)."""
    print("Loading preprocessed data (scaled)...")
    X_train = pd.read_csv(os.path.join(PROCESSED_DIR, "X_train.csv"))
    X_test = pd.read_csv(os.path.join(PROCESSED_DIR, "X_test.csv"))
    y_train = pd.read_csv(os.path.join(PROCESSED_DIR, "y_train.csv")).squeeze()
    y_test = pd.read_csv(os.path.join(PROCESSED_DIR, "y_test.csv")).squeeze()

    print(f"  Train: {X_train.shape[0]:,} samples, {X_train.shape[1]} features")
    print(f"  Test:  {X_test.shape[0]:,} samples")
    print(f"  Positive rate (train): {y_train.mean():.3f}")
    return X_train, X_test, y_train, y_test


def tune_and_train(X_train, y_train):
    """Hyperparameter tuning for MLP."""
    print("\n--- Hyperparameter Tuning (GridSearchCV) ---")
    print("  Note: MLP tuning can be slow — using focused search space.")

    param_grid = {
        "hidden_layer_sizes": [
            (128, 64, 32),
            (100, 50),
            (256, 128, 64),
            (64, 32),
        ],
        "activation": ["relu"],
        "alpha": [0.0001, 0.001, 0.01],
        "learning_rate": ["adaptive"],
        "learning_rate_init": [0.001, 0.01],
    }

    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    from sklearn.model_selection import RandomizedSearchCV

    grid = RandomizedSearchCV(
        MLPClassifier(
            max_iter=500,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=15,
            random_state=RANDOM_STATE,
            solver="adam",
            batch_size=256,
        ),
        param_grid, n_iter=12, cv=cv, scoring="roc_auc",
        n_jobs=-1, verbose=1, refit=True, random_state=RANDOM_STATE,
    )
    grid.fit(X_train, y_train)

    print(f"\n  Best AUC-ROC (CV): {grid.best_score_:.4f}")
    print(f"  Best parameters: {grid.best_params_}")
    return grid.best_estimator_, grid.best_params_


def cross_validate_model(model, X_train, y_train):
    """5-fold cross-validation metrics."""
    print(f"\n--- {N_FOLDS}-Fold Cross-Validation ---")
    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    y_proba = cross_val_predict(model, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)

    metrics = {
        "cv_auc_roc": roc_auc_score(y_train, y_proba),
        "cv_accuracy": accuracy_score(y_train, y_pred),
        "cv_precision": precision_score(y_train, y_pred),
        "cv_recall": recall_score(y_train, y_pred),
        "cv_f1": f1_score(y_train, y_pred),
        "cv_avg_precision": average_precision_score(y_train, y_proba),
    }

    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")
    return metrics


def evaluate_on_test(model, X_test, y_test):
    """Evaluate on hold-out test set."""
    print("\n--- Test Set Evaluation ---")
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)

    metrics = {
        "test_auc_roc": roc_auc_score(y_test, y_proba),
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_precision": precision_score(y_test, y_pred),
        "test_recall": recall_score(y_test, y_pred),
        "test_f1": f1_score(y_test, y_pred),
        "test_avg_precision": average_precision_score(y_test, y_proba),
    }

    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

    print(f"\n  Classification Report:")
    print(classification_report(y_test, y_pred,
          target_names=["No Readmit", "Readmit <30d"]))

    return metrics, y_pred, y_proba


def plot_training_loss(model):
    """Plot MLP training loss curve."""
    os.makedirs(PLOT_DIR, exist_ok=True)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(model.loss_curve_, color="#10b981", linewidth=1.5, label="Training Loss")
    if hasattr(model, "validation_scores_") and model.validation_scores_:
        ax2 = ax.twinx()
        ax2.plot(model.validation_scores_, color="#3b82f6", linewidth=1.5,
                 label="Validation AUC", linestyle="--")
        ax2.set_ylabel("Validation Score", color="#3b82f6")
        ax2.legend(loc="center right")

    ax.set_xlabel("Iteration")
    ax.set_ylabel("Loss", color="#10b981")
    ax.set_title("MLP Training Loss Curve")
    ax.legend(loc="upper right")
    ax.spines["top"].set_visible(False)

    plt.tight_layout()
    path = os.path.join(PLOT_DIR, "mlp_training_loss.png")
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")


def plot_roc_curve(y_test, y_proba):
    """Plot ROC curve."""
    os.makedirs(PLOT_DIR, exist_ok=True)
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(fpr, tpr, color="#10b981", linewidth=2, label=f"MLP (AUC = {auc:.4f})")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.5)
    ax.fill_between(fpr, tpr, alpha=0.1, color="#10b981")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curve — MLP Neural Network")
    ax.legend(loc="lower right")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    path = os.path.join(PLOT_DIR, "roc_curve.png")
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")


def plot_confusion_matrix(y_test, y_pred):
    """Plot confusion matrix."""
    os.makedirs(PLOT_DIR, exist_ok=True)
    cm = confusion_matrix(y_test, y_pred)

    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt=",d", cmap="Greens",
                xticklabels=["No Readmit", "Readmit <30d"],
                yticklabels=["No Readmit", "Readmit <30d"], ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title("Confusion Matrix — MLP Neural Network")

    plt.tight_layout()
    path = os.path.join(PLOT_DIR, "confusion_matrix.png")
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")


def main():
    print("=" * 60)
    print("MLP NEURAL NETWORK MODEL PIPELINE")
    print("Diabetes 30-Day Readmission Prediction")
    print("=" * 60)

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(PLOT_DIR, exist_ok=True)

    X_train, X_test, y_train, y_test = load_processed_data()

    best_model, best_params = tune_and_train(X_train, y_train)
    cv_metrics = cross_validate_model(best_model, X_train, y_train)
    test_metrics, y_pred, y_proba = evaluate_on_test(best_model, X_test, y_test)

    # Plots
    plot_training_loss(best_model)
    plot_roc_curve(y_test, y_proba)
    plot_confusion_matrix(y_test, y_pred)

    # Save
    joblib.dump(best_model, os.path.join(OUTPUT_DIR, "mlp_model.joblib"))
    all_metrics = {**cv_metrics, **test_metrics, "best_params": best_params}
    with open(os.path.join(OUTPUT_DIR, "metrics.json"), "w") as f:
        json.dump(all_metrics, f, indent=2, default=str)

    print(f"\n{'=' * 60}")
    print("MLP NEURAL NETWORK COMPLETE")
    print(f"  CV AUC-ROC:   {cv_metrics['cv_auc_roc']:.4f}")
    print(f"  Test AUC-ROC: {test_metrics['test_auc_roc']:.4f}")
    print(f"  Test F1:      {test_metrics['test_f1']:.4f}")
    print(f"  Outputs: {OUTPUT_DIR}/")
    print("=" * 60)


if __name__ == "__main__":
    main()

In [ ]:
"""
04_pipeline_diagram.py
======================
End-to-End ML Pipeline Diagram for Diabetes Readmission Prediction

Generates a professional pipeline diagram showing the full workflow
from data ingestion to model deployment.

Author: Jiahao Li
Team: Final Project - G4 (Abdullah, Jiahao, Feifan)
Course: MSDS 422 - Practical Machine Learning
Date: February 2026
"""

import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

OUTPUT_DIR = os.path.join("outputs", "pipeline")


def draw_pipeline_diagram():
    """Create the end-to-end ML pipeline diagram."""
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    fig, ax = plt.subplots(figsize=(20, 14))
    ax.set_xlim(0, 20)
    ax.set_ylim(0, 14)
    ax.axis("off")

    # Title
    ax.text(
        10, 13.5,
        "End-to-End ML Pipeline: Predicting 30-Day Diabetes Readmission",
        fontsize=18, fontweight="bold", ha="center", va="center",
        color="#1a1a2e",
    )
    ax.text(
        10, 13.0,
        "MSDS 422 Final Project — Team G4 (Abdullah, Jiahao, Feifan)",
        fontsize=11, ha="center", va="center", color="#555555",
    )

    # ================================================================
    # Color scheme
    # ================================================================
    colors = {
        "data": "#3b82f6",       # blue
        "preprocess": "#8b5cf6", # purple
        "feature": "#06b6d4",    # cyan
        "model": "#f59e0b",      # amber
        "eval": "#ef4444",       # red
        "deploy": "#10b981",     # green
        "arrow": "#374151",      # gray
    }

    def draw_box(x, y, w, h, color, title, items, alpha=0.15):
        """Draw a rounded box with title and bullet items."""
        box = FancyBboxPatch(
            (x, y), w, h,
            boxstyle="round,pad=0.15",
            facecolor=color, alpha=alpha,
            edgecolor=color, linewidth=2,
        )
        ax.add_patch(box)

        # Title
        ax.text(
            x + w / 2, y + h - 0.35,
            title, fontsize=12, fontweight="bold",
            ha="center", va="center", color=color,
        )

        # Items
        for i, item in enumerate(items):
            ax.text(
                x + 0.3, y + h - 0.75 - i * 0.35,
                f"• {item}", fontsize=8.5, va="center", color="#1a1a2e",
            )

    def draw_arrow(x1, y1, x2, y2, color="#374151"):
        """Draw a curved arrow between boxes."""
        arrow = FancyArrowPatch(
            (x1, y1), (x2, y2),
            arrowstyle="-|>",
            mutation_scale=15,
            linewidth=2,
            color=color,
            connectionstyle="arc3,rad=0.0",
        )
        ax.add_patch(arrow)

    # ================================================================
    # Row 1: Data Ingestion -> Preprocessing -> Feature Engineering
    # ================================================================
    row1_y = 9.5
    box_h = 2.8
    box_w = 5.2

    # Box 1: Data Ingestion
    draw_box(0.5, row1_y, box_w, box_h, colors["data"],
             "1. DATA INGESTION", [
                 "UCI Diabetes 130-US Hospitals",
                 "101,766 encounters, 55 features",
                 "Target: readmission < 30 days",
                 "Binary classification (11.2% positive)",
                 "Source: UCI ML Repository",
                 "Format: CSV (diabetic_data.csv)",
             ])

    # Arrow 1->2
    draw_arrow(5.7 + 0.2, row1_y + box_h / 2, 7.4 - 0.2, row1_y + box_h / 2)

    # Box 2: Data Preprocessing
    draw_box(7.4, row1_y, box_w, box_h, colors["preprocess"],
             "2. DATA PREPROCESSING", [
                 "Remove duplicates (first encounter only)",
                 "Filter invalid discharges (deceased/hospice)",
                 "Drop: weight (97% missing), payer_code (52%)",
                 "Impute: race (mode), specialty ('Unknown')",
                 "ICD-9 diagnosis grouping (9 categories)",
                 "~70K encounters after cleaning",
             ])

    # Arrow 2->3
    draw_arrow(12.6 + 0.2, row1_y + box_h / 2, 14.3 - 0.2, row1_y + box_h / 2)

    # Box 3: Feature Engineering
    draw_box(14.3, row1_y, box_w, box_h, colors["feature"],
             "3. FEATURE ENGINEERING", [
                 "Length-of-stay bins (short/med/long/ext)",
                 "Total prior visits (OP + ER + IP)",
                 "Visit intensity categories",
                 "Emergency/inpatient visit ratios",
                 "Procedure intensity (per day)",
                 "High-utilizer flag (3+ prior visits)",
                 "22 new features → 77 total",
             ])

    # ================================================================
    # Row 2: Model Training (4 models) -> Evaluation
    # ================================================================
    row2_y = 4.5
    model_w = 3.5
    model_h = 3.8

    # Arrow down from Feature Engineering to Models
    draw_arrow(17, row1_y - 0.1, 17, row2_y + model_h + 0.1)
    ax.text(17.5, row2_y + model_h + 0.5, "Train/Test\nSplit (80/20)",
            fontsize=8, ha="center", va="center", color="#555555",
            style="italic")

    # Box 4a: Logistic Regression (Abdullah)
    draw_box(0.3, row2_y, model_w, model_h, colors["model"],
             "4a. LOGISTIC REG.", [
                 "Owner: Abdullah",
                 "L2 regularization",
                 "Class weight: balanced",
                 "Baseline model",
                 "Interpretable coefficients",
                 "Scaled features required",
                 "Threshold optimization",
             ])

    # Box 4b: Random Forest (Abdullah)
    draw_box(4.1, row2_y, model_w, model_h, colors["model"],
             "4b. RANDOM FOREST", [
                 "Owner: Abdullah",
                 "n_estimators: 300",
                 "max_depth tuned (CV)",
                 "Class weight: balanced",
                 "Ensemble of decision trees",
                 "Built-in feature importance",
                 "No scaling needed",
             ])

    # Box 4c: XGBoost (Jiahao)
    draw_box(7.9, row2_y, model_w, model_h, colors["model"],
             "4c. XGBOOST ★", [
                 "Owner: Jiahao",
                 "Gradient boosting framework",
                 "scale_pos_weight for imbalance",
                 "GridSearchCV tuning:",
                 "  depth, lr, n_est, min_child",
                 "5-fold stratified CV",
                 "Regularization (L1 + L2)",
             ])

    # Box 4d: MLP Neural Network (Feifan)
    draw_box(11.7, row2_y, model_w, model_h, colors["model"],
             "4d. MLP (NEURAL NET)", [
                 "Owner: Feifan",
                 "Multi-layer perceptron",
                 "Hidden layers: (128, 64, 32)",
                 "ReLU activation + Dropout",
                 "Adam optimizer",
                 "Early stopping",
                 "Scaled features required",
             ])

    # Box 5: Evaluation
    draw_box(15.7, row2_y, 3.8, model_h, colors["eval"],
             "5. EVALUATION", [
                 "AUC-ROC (primary metric)",
                 "Precision / Recall / F1",
                 "Confusion matrix",
                 "ROC & PR curves",
                 "5-fold cross-validation",
                 "Model comparison table",
                 "Feature importance analysis",
             ])

    # Arrows from models to evaluation
    for x_start in [2.1, 5.9, 9.7, 13.5]:
        draw_arrow(x_start + model_w / 2, row2_y - 0.05,
                    17.6, row2_y - 0.05, color="#999999")

    # ================================================================
    # Row 3: Deployment
    # ================================================================
    row3_y = 1.0
    deploy_w = 18.5
    deploy_h = 2.2

    # Arrow down from evaluation
    draw_arrow(17.6, row2_y - 0.1, 10, row3_y + deploy_h + 0.15)

    draw_box(0.5, row3_y, deploy_w, deploy_h, colors["deploy"],
             "6. DEPLOYMENT & AUTOMATION", [
                 "Best model selection based on AUC-ROC  →  Export via joblib  →  "
                 "Automated pipeline (preprocessing → prediction)  →  "
                 "Dashboard (model comparison, feature importance, demographics)  →  "
                 "Monitoring & retraining schedule",
             ])

    # Additional deployment details
    ax.text(
        1.0, row3_y + 0.5,
        "Pipeline: Raw CSV → clean → engineer features → predict → output risk scores    |    "
        "Tools: Python, scikit-learn, XGBoost, pandas, matplotlib    |    "
        "Infra: GitHub repo, requirements.txt, automated scripts",
        fontsize=8, color="#555555", va="center",
    )

    # ================================================================
    # Legend
    # ================================================================
    legend_items = [
        mpatches.Patch(facecolor=colors["data"], alpha=0.3, label="Data Ingestion"),
        mpatches.Patch(facecolor=colors["preprocess"], alpha=0.3, label="Preprocessing"),
        mpatches.Patch(facecolor=colors["feature"], alpha=0.3, label="Feature Engineering"),
        mpatches.Patch(facecolor=colors["model"], alpha=0.3, label="Model Training"),
        mpatches.Patch(facecolor=colors["eval"], alpha=0.3, label="Evaluation"),
        mpatches.Patch(facecolor=colors["deploy"], alpha=0.3, label="Deployment"),
    ]
    ax.legend(
        handles=legend_items, loc="upper right",
        fontsize=9, framealpha=0.9, ncol=3,
        bbox_to_anchor=(0.98, 0.98),
    )

    plt.tight_layout()

    # Save
    filepath = os.path.join(OUTPUT_DIR, "ml_pipeline_diagram.png")
    plt.savefig(filepath, dpi=200, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"Pipeline diagram saved: {filepath}")

    # Also save as PDF for report
    filepath_pdf = os.path.join(OUTPUT_DIR, "ml_pipeline_diagram.pdf")
    fig2, ax2 = plt.subplots(figsize=(20, 14))
    ax2.axis("off")
    # Re-render for PDF (reuse the same function structure)
    plt.close()

    return filepath


def main():
    """Generate pipeline diagram."""
    print("=" * 60)
    print("GENERATING ML PIPELINE DIAGRAM")
    print("=" * 60)
    filepath = draw_pipeline_diagram()
    print(f"\nDone! Output: {filepath}")
    print("Upload this to your GitHub repo under outputs/ or docs/")


if __name__ == "__main__":
    main()

In [ ]:
"""
05_model_comparison.py
======================
Model Comparison and Final Evaluation Dashboard

Loads metrics from all 4 models, creates:
- Comparison table (CSV + formatted console output)
- Combined ROC curves (all models on one plot)
- Combined Precision-Recall curves
- Performance bar charts
- Final recommendation summary

Author: Abdullah Abdulsami (comparison table), Jiahao Li (visualizations)
Team: Final Project - G4 (Abdullah, Jiahao, Feifan)
Course: MSDS 422 - Practical Machine Learning
Date: February 2026
"""

import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score
import joblib

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 150
plt.rcParams["font.size"] = 11

# ============================================================
# Configuration
# ============================================================
PROCESSED_DIR = os.path.join("data", "processed")
OUTPUT_DIR = os.path.join("outputs", "comparison")
PLOT_DIR = os.path.join(OUTPUT_DIR, "plots")

MODELS = {
    "Logistic Regression": {
        "metrics_path": os.path.join("outputs", "logistic_regression", "metrics.json"),
        "model_path": os.path.join("outputs", "logistic_regression", "logistic_regression_model.joblib"),
        "color": "#8b5cf6",
        "scaled": True,
        "owner": "Abdullah",
    },
    "Random Forest": {
        "metrics_path": os.path.join("outputs", "random_forest", "metrics.json"),
        "model_path": os.path.join("outputs", "random_forest", "random_forest_model.joblib"),
        "color": "#f59e0b",
        "scaled": False,
        "owner": "Abdullah",
    },
    "XGBoost": {
        "metrics_path": os.path.join("outputs", "xgboost", "metrics.json"),
        "model_path": os.path.join("outputs", "xgboost", "xgboost_best_model.joblib"),
        "color": "#2563eb",
        "scaled": False,
        "owner": "Jiahao",
    },
    "MLP Neural Network": {
        "metrics_path": os.path.join("outputs", "mlp", "metrics.json"),
        "model_path": os.path.join("outputs", "mlp", "mlp_model.joblib"),
        "color": "#10b981",
        "scaled": True,
        "owner": "Feifan",
    },
}


def load_test_data():
    """Load test data for generating curves."""
    X_test_scaled = pd.read_csv(os.path.join(PROCESSED_DIR, "X_test.csv"))
    X_test_unscaled = pd.read_csv(os.path.join(PROCESSED_DIR, "X_test_unscaled.csv"))
    y_test = pd.read_csv(os.path.join(PROCESSED_DIR, "y_test.csv")).squeeze()
    return X_test_scaled, X_test_unscaled, y_test


def load_all_metrics():
    """Load metrics from each model's JSON output."""
    print("\n--- Loading Model Metrics ---")
    results = {}

    for name, config in MODELS.items():
        path = config["metrics_path"]
        if os.path.exists(path):
            with open(path) as f:
                results[name] = json.load(f)
            print(f"  Loaded: {name}")
        else:
            print(f"  WARNING: {name} metrics not found at {path}")
            print(f"           Run the model script first.")

    return results


def create_comparison_table(results):
    """Create formatted comparison table."""
    print("\n--- Model Comparison Table ---\n")

    metrics_keys = [
        ("test_auc_roc", "AUC-ROC"),
        ("test_accuracy", "Accuracy"),
        ("test_precision", "Precision"),
        ("test_recall", "Recall"),
        ("test_f1", "F1 Score"),
        ("test_avg_precision", "Avg Precision"),
        ("cv_auc_roc", "CV AUC-ROC"),
        ("cv_f1", "CV F1"),
    ]

    rows = []
    for name, metrics in results.items():
        row = {"Model": name, "Owner": MODELS[name]["owner"]}
        for key, label in metrics_keys:
            row[label] = metrics.get(key, np.nan)
        rows.append(row)

    df = pd.DataFrame(rows)

    # Sort by AUC-ROC descending
    df = df.sort_values("AUC-ROC", ascending=False).reset_index(drop=True)

    # Add rank
    df.insert(0, "Rank", range(1, len(df) + 1))

    # Print formatted
    print(df.to_string(index=False, float_format="%.4f"))

    # Save CSV
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    csv_path = os.path.join(OUTPUT_DIR, "model_comparison_table.csv")
    df.to_csv(csv_path, index=False)
    print(f"\n  Saved: {csv_path}")

    # Identify best model
    best = df.iloc[0]
    print(f"\n  BEST MODEL: {best['Model']} (AUC-ROC: {best['AUC-ROC']:.4f})")

    return df


def plot_combined_roc(y_test, models_data):
    """Plot all model ROC curves on one figure."""
    os.makedirs(PLOT_DIR, exist_ok=True)

    fig, ax = plt.subplots(figsize=(9, 7))

    for name, data in models_data.items():
        if data["y_proba"] is not None:
            fpr, tpr, _ = roc_curve(y_test, data["y_proba"])
            auc = roc_auc_score(y_test, data["y_proba"])
            ax.plot(fpr, tpr, color=MODELS[name]["color"], linewidth=2,
                    label=f"{name} (AUC = {auc:.4f})")

    ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Random (AUC = 0.500)")
    ax.set_xlabel("False Positive Rate", fontsize=12)
    ax.set_ylabel("True Positive Rate", fontsize=12)
    ax.set_title("ROC Curve Comparison — All Models", fontsize=14, fontweight="bold")
    ax.legend(loc="lower right", fontsize=10)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    path = os.path.join(PLOT_DIR, "combined_roc_curves.png")
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")


def plot_combined_pr(y_test, models_data):
    """Plot all model Precision-Recall curves."""
    os.makedirs(PLOT_DIR, exist_ok=True)

    fig, ax = plt.subplots(figsize=(9, 7))

    for name, data in models_data.items():
        if data["y_proba"] is not None:
            precision, recall, _ = precision_recall_curve(y_test, data["y_proba"])
            ax.plot(recall, precision, color=MODELS[name]["color"], linewidth=2,
                    label=f"{name}")

    baseline = y_test.mean()
    ax.axhline(y=baseline, color="gray", linestyle="--", alpha=0.5,
               label=f"Baseline ({baseline:.3f})")
    ax.set_xlabel("Recall", fontsize=12)
    ax.set_ylabel("Precision", fontsize=12)
    ax.set_title("Precision-Recall Curve Comparison", fontsize=14, fontweight="bold")
    ax.legend(loc="upper right", fontsize=10)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    path = os.path.join(PLOT_DIR, "combined_pr_curves.png")
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")


def plot_metrics_barchart(comparison_df):
    """Bar chart comparing key metrics across models."""
    os.makedirs(PLOT_DIR, exist_ok=True)

    metrics_to_plot = ["AUC-ROC", "F1 Score", "Precision", "Recall"]
    models = comparison_df["Model"].values
    colors = [MODELS[m]["color"] for m in models]

    fig, axes = plt.subplots(1, 4, figsize=(18, 5))

    for idx, metric in enumerate(metrics_to_plot):
        values = comparison_df[metric].values
        bars = axes[idx].bar(range(len(models)), values, color=colors, width=0.6)
        axes[idx].set_xticks(range(len(models)))
        axes[idx].set_xticklabels([m.replace(" ", "\n") for m in models], fontsize=8)
        axes[idx].set_title(metric, fontsize=12, fontweight="bold")
        axes[idx].set_ylim(0, max(values) * 1.2)
        axes[idx].spines["top"].set_visible(False)
        axes[idx].spines["right"].set_visible(False)

        # Value labels
        for bar, val in zip(bars, values):
            axes[idx].text(
                bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9,
            )

    plt.suptitle("Model Performance Comparison", fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    path = os.path.join(PLOT_DIR, "metrics_comparison_barchart.png")
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")


def generate_summary(comparison_df, results):
    """Generate text summary for report."""
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    best = comparison_df.iloc[0]
    summary = []
    summary.append("=" * 60)
    summary.append("MODEL COMPARISON SUMMARY")
    summary.append("Predicting 30-Day Diabetes Readmission")
    summary.append("=" * 60)
    summary.append("")
    summary.append(f"Best Performing Model: {best['Model']}")
    summary.append(f"  Owner: {best['Owner']}")
    summary.append(f"  Test AUC-ROC: {best['AUC-ROC']:.4f}")
    summary.append(f"  Test F1 Score: {best['F1 Score']:.4f}")
    summary.append(f"  Test Precision: {best['Precision']:.4f}")
    summary.append(f"  Test Recall: {best['Recall']:.4f}")
    summary.append("")
    summary.append("All Models Ranked:")
    for _, row in comparison_df.iterrows():
        summary.append(
            f"  {row['Rank']}. {row['Model']:25s} | AUC: {row['AUC-ROC']:.4f} "
            f"| F1: {row['F1 Score']:.4f} | Owner: {row['Owner']}"
        )
    summary.append("")
    summary.append("Key Findings:")
    summary.append("- Class imbalance (11.2% positive) is a major challenge")
    summary.append("- AUC-ROC is the primary metric (robust to imbalance)")
    summary.append("- Gradient boosting methods handle sparse clinical features well")
    summary.append("- Feature importance reveals prior utilization as top predictor")
    summary.append("")
    summary.append("Recommendation:")
    summary.append(f"  Deploy {best['Model']} as the production model.")
    summary.append("  Monitor for data drift in medication patterns and")
    summary.append("  readmission definitions. Retrain quarterly.")
    summary.append("=" * 60)

    text = "\n".join(summary)
    print(text)

    path = os.path.join(OUTPUT_DIR, "summary.txt")
    with open(path, "w") as f:
        f.write(text)
    print(f"\n  Saved: {path}")


def main():
    print("=" * 60)
    print("MODEL COMPARISON PIPELINE")
    print("Diabetes 30-Day Readmission Prediction")
    print("=" * 60)

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(PLOT_DIR, exist_ok=True)

    # Load metrics
    results = load_all_metrics()

    if len(results) == 0:
        print("\nERROR: No model metrics found. Run the individual model scripts first:")
        print("  python scripts/03a_logistic_regression.py")
        print("  python scripts/03b_random_forest.py")
        print("  python scripts/03c_xgboost_model.py")
        print("  python scripts/03d_mlp_neural_network.py")
        return

    # Comparison table
    comparison_df = create_comparison_table(results)

    # Load test data and models for curve plots
    print("\n--- Loading Models for Curve Generation ---")
    X_test_scaled, X_test_unscaled, y_test = load_test_data()

    models_data = {}
    for name, config in MODELS.items():
        if os.path.exists(config["model_path"]):
            model = joblib.load(config["model_path"])
            X = X_test_scaled if config["scaled"] else X_test_unscaled
            y_proba = model.predict_proba(X)[:, 1]
            models_data[name] = {"model": model, "y_proba": y_proba}
            print(f"  Loaded: {name}")
        else:
            models_data[name] = {"model": None, "y_proba": None}
            print(f"  SKIP: {name} (model file not found)")

    # Plots
    print("\n--- Generating Comparison Plots ---")
    plot_combined_roc(y_test, models_data)
    plot_combined_pr(y_test, models_data)
    plot_metrics_barchart(comparison_df)

    # Summary
    generate_summary(comparison_df, results)

    print(f"\n{'=' * 60}")
    print("MODEL COMPARISON COMPLETE")
    print(f"  Outputs: {OUTPUT_DIR}/")
    print("=" * 60)


if __name__ == "__main__":
    main()

In [ ]:
"""
06_run_pipeline.py
==================
End-to-End Automated ML Pipeline Runner

Executes the complete workflow:
  1. Feature Engineering (preprocessing + feature creation)
  2. Model Training (all 4 models)
  3. Model Comparison (evaluation + visualization)
  4. Pipeline Diagram generation

Usage:
    python scripts/06_run_pipeline.py              # Run everything
    python scripts/06_run_pipeline.py --models-only # Skip preprocessing
    python scripts/06_run_pipeline.py --compare-only # Only comparison step

Author: Jiahao Li (pipeline), Abdullah (scripts), Feifan (documentation)
Team: Final Project - G4
Course: MSDS 422 - Practical Machine Learning
Date: February 2026
"""

import os
import sys
import time
import subprocess
import argparse
from datetime import datetime


SCRIPTS = {
    "Feature Engineering": "scripts/02_feature_engineering.py",
    "Logistic Regression": "scripts/03a_logistic_regression.py",
    "Random Forest": "scripts/03b_random_forest.py",
    "XGBoost": "scripts/03c_xgboost_model.py",
    "MLP Neural Network": "scripts/03d_mlp_neural_network.py",
    "Model Comparison": "scripts/05_model_comparison.py",
    "Pipeline Diagram": "scripts/04_pipeline_diagram.py",
}


def run_script(name, path):
    """Run a single Python script and report status."""
    print(f"\n{'─' * 60}")
    print(f"  RUNNING: {name}")
    print(f"  Script:  {path}")
    print(f"{'─' * 60}\n")

    if not os.path.exists(path):
        print(f"  ERROR: Script not found at {path}")
        return False, 0

    start = time.time()
    result = subprocess.run(
        [sys.executable, path],
        capture_output=False,
    )
    elapsed = time.time() - start

    if result.returncode == 0:
        print(f"\n  SUCCESS: {name} completed in {elapsed:.1f}s")
        return True, elapsed
    else:
        print(f"\n  FAILED: {name} (exit code {result.returncode})")
        return False, elapsed


def main():
    parser = argparse.ArgumentParser(description="Run ML Pipeline")
    parser.add_argument("--models-only", action="store_true",
                        help="Skip preprocessing, run models + comparison only")
    parser.add_argument("--compare-only", action="store_true",
                        help="Only run comparison step (models must be trained)")
    args = parser.parse_args()

    print("=" * 60)
    print("AUTOMATED ML PIPELINE")
    print("Predicting 30-Day Diabetes Readmission")
    print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 60)

    # Determine which steps to run
    if args.compare_only:
        steps = ["Model Comparison"]
    elif args.models_only:
        steps = [
            "Logistic Regression", "Random Forest",
            "XGBoost", "MLP Neural Network",
            "Model Comparison", "Pipeline Diagram",
        ]
    else:
        steps = list(SCRIPTS.keys())

    # Check prerequisites
    data_file = os.path.join("data", "diabetic_data.csv")
    if not args.compare_only and not args.models_only:
        if not os.path.exists(data_file):
            print(f"\n  ERROR: Dataset not found at {data_file}")
            print("  Please download from UCI ML Repository first.")
            print("  See data/README.md for instructions.")
            sys.exit(1)

    if args.models_only or args.compare_only:
        processed = os.path.join("data", "processed", "X_train.csv")
        if not os.path.exists(processed):
            print(f"\n  ERROR: Processed data not found. Run full pipeline first.")
            sys.exit(1)

    # Run steps
    results = {}
    total_start = time.time()

    for step_name in steps:
        script_path = SCRIPTS[step_name]
        success, elapsed = run_script(step_name, script_path)
        results[step_name] = {"success": success, "time": elapsed}

        if not success and step_name == "Feature Engineering":
            print("\n  ABORTING: Preprocessing failed. Cannot continue.")
            sys.exit(1)

    total_time = time.time() - total_start

    # Summary
    print("\n" + "=" * 60)
    print("PIPELINE EXECUTION SUMMARY")
    print("=" * 60)
    print(f"\n  {'Step':<25s} {'Status':<10s} {'Time':>8s}")
    print(f"  {'─' * 45}")

    for step_name, result in results.items():
        status = "PASS" if result["success"] else "FAIL"
        print(f"  {step_name:<25s} {status:<10s} {result['time']:>7.1f}s")

    passed = sum(1 for r in results.values() if r["success"])
    total = len(results)

    print(f"\n  Total: {passed}/{total} steps passed")
    print(f"  Total time: {total_time:.1f}s ({total_time/60:.1f} min)")
    print(f"\n  Output directories:")
    print(f"    data/processed/           - Preprocessed datasets")
    print(f"    outputs/logistic_regression/ - LR model + plots")
    print(f"    outputs/random_forest/       - RF model + plots")
    print(f"    outputs/xgboost/             - XGBoost model + plots")
    print(f"    outputs/mlp/                 - MLP model + plots")
    print(f"    outputs/comparison/          - Comparison table + plots")
    print(f"    outputs/pipeline/            - Pipeline diagram")
    print(f"\n{'=' * 60}")
    print("PIPELINE COMPLETE")
    print("=" * 60)


if __name__ == "__main__":
    main()